# Eval Results Summary Tables

`eval_results_sunghyun` 아래의 raw CSV / timing CSV를 직접 읽어서 표를 만듭니다.

- `QUAL`, `LEAK` 평균은 raw eval CSV에서 **음수 점수를 제외한 뒤 다시 계산**합니다.
- timing 표는 모델별로 `Before`, `After`, `Delta (After - Before)` 평균 시간을 보여줍니다.
- `Delta LEAK`는 음수일수록 leakage 감소가 크다는 뜻입니다.


In [8]:
from __future__ import annotations

import csv
import html
import math
from pathlib import Path
from statistics import mean

try:
    from IPython.display import HTML, display
except ModuleNotFoundError:
    class HTML(str):
        pass

    def display(obj):
        print(obj)

DISPLAY_RUNS = [
    {
        "key": "llama31_paper",
        "display_name": "Llama-3.1-8B-Instruct (Paper)",
        "static_score_row": {
            "model": "Llama-3.1-8B-Instruct (Paper)",
            "before_qual": 0.718,
            "before_leak": 0.174,
            "after_qual": 0.855,
            "after_leak": 0.075,
            "delta_qual": 0.137,
            "delta_leak": -0.099,
        },
        "score_files": {},
        "timing_files": {},
    },
    {
        "key": "llama31_ours",
        "display_name": "Llama-3.1-8B-Instruct (Ours)",
        "score_files": {
            "before": Path("eval_results_sunghyun/eval_llama31_8b_PUPA_TNB_before.csv"),
            "after": Path("eval_results_sunghyun/eval_llama31_8b_PUPA_TNB_after.csv"),
        },
        "timing_files": {
            "before": Path("eval_results_sunghyun/eval_llama31_8b_PUPA_TNB_before_timing.csv"),
            "after": Path("eval_results_sunghyun/eval_llama31_8b_PUPA_TNB_after_timing.csv"),
        },
    },
    {
        "key": "gemma3_4b",
        "display_name": "Gemma-3-4B",
        "score_files": {
            "before": Path("eval_results_sunghyun/eval_gemma3_4b_PUPA_TNB_before.csv"),
            "after": Path("eval_results_sunghyun/eval_gemma3_4b_PUPA_TNB_after.csv"),
        },
        "timing_files": {
            "before": Path("eval_results_sunghyun/eval_gemma3_4b_PUPA_TNB_before_timing.csv"),
            "after": Path("eval_results_sunghyun/eval_gemma3_4b_PUPA_TNB_after_timing.csv"),
        },
    },
    {
        "key": "qwen35_9b_nothink",
        "display_name": "Qwen3.5-9B-nothink",
        "score_files": {
            "before": Path("eval_results_sunghyun/eval_qwen35_9b_nothink_PUPA_TNB_before.csv"),
            "after": Path("eval_results_sunghyun/eval_qwen35_9b_nothink_PUPA_TNB_after.csv"),
        },
        "timing_files": {
            "before": Path("eval_results_sunghyun/eval_qwen35_9b_nothink_PUPA_TNB_before_timing.csv"),
            "after": Path("eval_results_sunghyun/eval_qwen35_9b_nothink_PUPA_TNB_after_timing.csv"),
        },
    },
]

TIMING_ALIASES = {
    "stage1_sec": ["stage1_prompt_creation", "stage1_prompt_creation_sec"],
    "stage2a_sec": ["stage2a_cloud_response", "stage2a_cloud_response_sec"],
    "stage2b_sec": ["stage2b_local_aggregation", "stage2b_local_aggregation_sec"],
    "judge_sec": ["judge_evaluation", "judge_evaluation_sec"],
    "pipeline_sec": ["total_pipeline", "total_pipeline_sec"],
    "total_row_sec": ["total_row", "time_total_row_sec"],
}

TABLE_STYLE = """
<style>
.eval-table {
    border-collapse: collapse;
    font-family: "Times New Roman", Georgia, serif;
    font-size: 19px;
    line-height: 1.2;
    margin: 18px 0;
    min-width: 960px;
}
.eval-table caption {
    caption-side: top;
    font-size: 24px;
    font-weight: 700;
    text-align: left;
    margin-bottom: 8px;
}
.eval-table th,
.eval-table td {
    padding: 6px 14px;
    text-align: center;
    vertical-align: middle;
    white-space: nowrap;
}
.eval-table thead tr:first-child th {
    border-top: 3px solid #111;
}
.eval-table thead tr:last-child th {
    border-bottom: 2px solid #111;
}
.eval-table tbody tr:last-child td {
    border-bottom: 3px solid #111;
}
.eval-table .row-header {
    text-align: left;
}
.eval-table .best {
    font-weight: 700;
}
.eval-table .na {
    color: #6b7280;
}
.eval-note {
    color: #374151;
    font-size: 14px;
    margin-top: -6px;
}
</style>
"""


def safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def find_first_column(fieldnames, candidates):
    lowered = {name.lower(): name for name in fieldnames}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    return None


def read_csv_rows(path: Path):
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        return reader.fieldnames or [], list(reader)


def nonnegative_mean(rows, column_name):
    values = []
    excluded_negative = 0
    for row in rows:
        value = safe_float(row.get(column_name))
        if value is None:
            continue
        if value < 0:
            excluded_negative += 1
            continue
        values.append(value)
    return {
        "mean": mean(values) if values else None,
        "count": len(values),
        "excluded_negative": excluded_negative,
    }


def summarize_score_csv(path: Path):
    fieldnames, rows = read_csv_rows(path)
    qual_col = find_first_column(fieldnames, ["quals", "qual", "quality"])
    leak_col = find_first_column(fieldnames, ["leaks", "leak", "leakage"])
    return {
        "qual": nonnegative_mean(rows, qual_col) if qual_col else None,
        "leak": nonnegative_mean(rows, leak_col) if leak_col else None,
    }


def summarize_timing_csv(path: Path):
    fieldnames, rows = read_csv_rows(path)
    usable_rows = []
    for row in rows:
        status = (row.get("status") or "ok").strip().lower()
        if status not in ("", "ok"):
            continue
        usable_rows.append(row)

    def values_for(candidates):
        column_name = find_first_column(fieldnames, candidates)
        if not column_name:
            return []
        values = []
        for row in usable_rows:
            value = safe_float(row.get(column_name))
            if value is not None:
                values.append(value)
        return values

    summary = {}
    for key, aliases in TIMING_ALIASES.items():
        values = values_for(aliases)
        summary[key] = mean(values) if values else None

    if summary["pipeline_sec"] is None:
        stage_cols = [find_first_column(fieldnames, TIMING_ALIASES[key]) for key in ("stage1_sec", "stage2a_sec", "stage2b_sec")]
        if all(stage_cols):
            pipeline_values = []
            for row in usable_rows:
                stage_values = [safe_float(row.get(column)) for column in stage_cols]
                if all(value is not None for value in stage_values):
                    pipeline_values.append(sum(stage_values))
            summary["pipeline_sec"] = mean(pipeline_values) if pipeline_values else None

    if summary["total_row_sec"] is None:
        total_cols = [find_first_column(fieldnames, TIMING_ALIASES[key]) for key in ("stage1_sec", "stage2a_sec", "stage2b_sec", "judge_sec")]
        if all(total_cols):
            total_values = []
            for row in usable_rows:
                row_values = [safe_float(row.get(column)) for column in total_cols]
                if all(value is not None for value in row_values):
                    total_values.append(sum(row_values))
            summary["total_row_sec"] = mean(total_values) if total_values else None

    summary["row_count"] = len(usable_rows)
    return summary


def collect_results(run_specs):
    score_runs = {}
    timing_runs = {}

    for spec in run_specs:
        score_runs[spec["key"]] = {}
        timing_runs[spec["key"]] = {}

        for phase, path in spec.get("score_files", {}).items():
            if path and path.exists():
                score_runs[spec["key"]][phase] = summarize_score_csv(path)

        for phase, path in spec.get("timing_files", {}).items():
            if path and path.exists():
                timing_runs[spec["key"]][phase] = summarize_timing_csv(path)

    return score_runs, timing_runs


def format_score(value, signed=False):
    if value is None:
        return '<span class="na">N/A</span>'
    scaled = value * 100.0
    return f"{scaled:+.1f}" if signed else f"{scaled:.1f}"


def format_seconds(value, signed=False):
    if value is None:
        return '<span class="na">N/A</span>'
    return f"{value:+.2f}" if signed else f"{value:.2f}"


def is_best(value, target):
    return value is not None and target is not None and math.isclose(value, target, rel_tol=1e-9, abs_tol=1e-12)


def render_score_table(rows):
    metric_keys = [
        ("before_qual", max),
        ("before_leak", min),
        ("after_qual", max),
        ("after_leak", min),
        ("delta_qual", max),
        ("delta_leak", min),
    ]
    best_values = {}
    for key, reducer in metric_keys:
        values = [row[key] for row in rows if row[key] is not None]
        best_values[key] = reducer(values) if values else None

    parts = [TABLE_STYLE, '<table class="eval-table">']
    parts.append("<caption>QUAL / LEAK Summary</caption>")
    parts.append("<thead>")
    parts.append("<tr><th rowspan='2' class='row-header'>Model</th><th colspan='2'>Before Optimization</th><th colspan='2'>After Optimization</th><th colspan='2'>Difference</th></tr>")
    parts.append("<tr><th>QUAL ↑</th><th>LEAK ↓</th><th>QUAL ↑</th><th>LEAK ↓</th><th>ΔQUAL ↑</th><th>ΔLEAK ↓</th></tr>")
    parts.append("</thead><tbody>")

    for row in rows:
        parts.append("<tr>")
        parts.append(f"<td class='row-header'>{html.escape(row['model'])}</td>")
        for key, signed in [
            ("before_qual", False),
            ("before_leak", False),
            ("after_qual", False),
            ("after_leak", False),
            ("delta_qual", True),
            ("delta_leak", True),
        ]:
            classes = []
            if is_best(row[key], best_values[key]):
                classes.append("best")
            class_attr = f" class='{' '.join(classes)}'" if classes else ""
            parts.append(f"<td{class_attr}>{format_score(row[key], signed=signed)}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table>")
    parts.append("<div class='eval-note'>Llama-3.1-8B-Instruct (Paper) row uses the numbers from the attached reference image. Other rows are computed from local CSV files, with negative QUAL / LEAK scores excluded before averaging.</div>")
    return HTML("".join(parts))


def render_simple_table(title, columns, rows, best_mode="min", signed=False):
    best_values = {}
    for key, _ in columns[1:]:
        values = [row[key] for row in rows if row[key] is not None]
        if not values:
            best_values[key] = None
        elif best_mode == "max":
            best_values[key] = max(values)
        else:
            best_values[key] = min(values)

    parts = [TABLE_STYLE, '<table class="eval-table">']
    parts.append(f"<caption>{html.escape(title)}</caption>")
    parts.append("<thead><tr>")
    for _, label in columns:
        class_name = "row-header" if label == "Model" else ""
        parts.append(f"<th class='{class_name}'>{html.escape(label)}</th>")
    parts.append("</tr></thead><tbody>")

    for row in rows:
        parts.append("<tr>")
        for index, (key, _) in enumerate(columns):
            if index == 0:
                parts.append(f"<td class='row-header'>{html.escape(row[key])}</td>")
                continue
            classes = []
            if is_best(row[key], best_values[key]):
                classes.append("best")
            class_attr = f" class='{' '.join(classes)}'" if classes else ""
            parts.append(f"<td{class_attr}>{format_seconds(row[key], signed=signed)}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table>")
    return HTML("".join(parts))


score_runs, timing_runs = collect_results(DISPLAY_RUNS)
print(f"Loaded {len(DISPLAY_RUNS)} configured model rows")



Loaded 4 configured model rows


In [9]:
score_rows = []
exclusion_rows = []

for spec in DISPLAY_RUNS:
    model_name = spec["display_name"]
    static_score_row = spec.get("static_score_row")

    if static_score_row is not None:
        score_rows.append(dict(static_score_row))
        for phase in ("Before", "After"):
            exclusion_rows.append(
                {
                    "model": model_name,
                    "phase": phase,
                    "qual_used": "N/A",
                    "qual_excluded": "N/A",
                    "leak_used": "N/A",
                    "leak_excluded": "N/A",
                }
            )
        continue

    before = score_runs.get(spec["key"], {}).get("before", {})
    after = score_runs.get(spec["key"], {}).get("after", {})

    before_qual = before.get("qual", {}).get("mean") if before.get("qual") else None
    before_leak = before.get("leak", {}).get("mean") if before.get("leak") else None
    after_qual = after.get("qual", {}).get("mean") if after.get("qual") else None
    after_leak = after.get("leak", {}).get("mean") if after.get("leak") else None

    score_rows.append(
        {
            "model": model_name,
            "before_qual": before_qual,
            "before_leak": before_leak,
            "after_qual": after_qual,
            "after_leak": after_leak,
            "delta_qual": (after_qual - before_qual) if None not in (before_qual, after_qual) else None,
            "delta_leak": (after_leak - before_leak) if None not in (before_leak, after_leak) else None,
        }
    )

    for phase, run in (("Before", before), ("After", after)):
        if not run:
            exclusion_rows.append(
                {
                    "model": model_name,
                    "phase": phase,
                    "qual_used": "N/A",
                    "qual_excluded": "N/A",
                    "leak_used": "N/A",
                    "leak_excluded": "N/A",
                }
            )
            continue
        exclusion_rows.append(
            {
                "model": model_name,
                "phase": phase,
                "qual_used": run["qual"]["count"] if run.get("qual") else None,
                "qual_excluded": run["qual"]["excluded_negative"] if run.get("qual") else None,
                "leak_used": run["leak"]["count"] if run.get("leak") else None,
                "leak_excluded": run["leak"]["excluded_negative"] if run.get("leak") else None,
            }
        )


display(render_score_table(score_rows))

exclusion_columns = [
    ("model", "Model"),
    ("phase", "Stage"),
    ("qual_used", "QUAL Used Rows"),
    ("qual_excluded", "QUAL Negatives Excluded"),
    ("leak_used", "LEAK Used Rows"),
    ("leak_excluded", "LEAK Negatives Excluded"),
]


def render_count_table(title, columns, rows):
    parts = [TABLE_STYLE, '<table class="eval-table">']
    parts.append(f"<caption>{html.escape(title)}</caption>")
    parts.append("<thead><tr>")
    for _, label in columns:
        class_name = "row-header" if label in {"Model", "Stage"} else ""
        parts.append(f"<th class='{class_name}'>{html.escape(label)}</th>")
    parts.append("</tr></thead><tbody>")
    for row in rows:
        parts.append("<tr>")
        for key, label in columns:
            value = row[key]
            class_name = "row-header" if label in {"Model", "Stage"} else ""
            class_attr = f" class='{class_name}'" if class_name else ""
            parts.append(f"<td{class_attr}>{html.escape(str(value))}</td>")
        parts.append("</tr>")
    parts.append("</tbody></table>")
    return HTML("".join(parts))


display(render_count_table("Negative-Score Filtering Summary", exclusion_columns, exclusion_rows))



Model,Stage,QUAL Used Rows,QUAL Negatives Excluded,LEAK Used Rows,LEAK Negatives Excluded
Llama-3.1-8B-Instruct (Paper),Before,N/A,N/A,N/A,N/A
Llama-3.1-8B-Instruct (Paper),After,N/A,N/A,N/A,N/A
Llama-3.1-8B-Instruct (Ours),Before,236,0,236,0
Llama-3.1-8B-Instruct (Ours),After,236,0,236,0
Gemma-3-4B,Before,235,0,235,0
Gemma-3-4B,After,235,0,235,0
Qwen3.5-9B-nothink,Before,236,1,236,1
Qwen3.5-9B-nothink,After,236,1,236,1


In [10]:
TIMING_COLUMNS = [
    ("model", "Model"),
    ("stage1_sec", "Stage 1"),
    ("stage2a_sec", "Stage 2a"),
    ("stage2b_sec", "Stage 2b"),
    ("judge_sec", "Judge"),
    ("pipeline_sec", "Pipeline"),
    ("total_row_sec", "Total"),
]

before_timing_rows = []
after_timing_rows = []
delta_timing_rows = []

for spec in DISPLAY_RUNS:
    model_name = spec["display_name"]
    before = timing_runs.get(spec["key"], {}).get("before")
    after = timing_runs.get(spec["key"], {}).get("after")

    before_timing_rows.append({"model": model_name, **{key: (before.get(key) if before else None) for key, _ in TIMING_COLUMNS[1:]}})
    after_timing_rows.append({"model": model_name, **{key: (after.get(key) if after else None) for key, _ in TIMING_COLUMNS[1:]}})
    delta_timing_rows.append(
        {
            "model": model_name,
            **{
                key: (after.get(key) - before.get(key)) if before and after and None not in (before.get(key), after.get(key)) else None
                for key, _ in TIMING_COLUMNS[1:]
            },
        }
    )


display(render_simple_table("Timing Means By Model: Before (sec)", TIMING_COLUMNS, before_timing_rows, best_mode="min", signed=False))
display(render_simple_table("Timing Means By Model: After (sec)", TIMING_COLUMNS, after_timing_rows, best_mode="min", signed=False))
display(render_simple_table("Timing Delta By Model: After - Before (sec)", TIMING_COLUMNS, delta_timing_rows, best_mode="min", signed=True))



Model,Stage 1,Stage 2a,Stage 2b,Judge,Pipeline,Total
Llama-3.1-8B-Instruct (Paper),N/A,N/A,N/A,N/A,N/A,N/A
Llama-3.1-8B-Instruct (Ours),3.12,6.90,6.60,11.54,16.62,28.16
Gemma-3-4B,7.91,7.49,18.78,11.79,34.18,45.96
Qwen3.5-9B-nothink,40.45,6.86,50.60,11.98,97.15,109.66


Model,Stage 1,Stage 2a,Stage 2b,Judge,Pipeline,Total
Llama-3.1-8B-Instruct (Paper),N/A,N/A,N/A,N/A,N/A,N/A
Llama-3.1-8B-Instruct (Ours),3.12,7.90,8.29,11.55,19.30,30.86
Gemma-3-4B,2.47,7.12,3.76,9.85,13.35,23.21
Qwen3.5-9B-nothink,41.89,6.32,48.96,11.52,96.28,108.47


Model,Stage 1,Stage 2a,Stage 2b,Judge,Pipeline,Total
Llama-3.1-8B-Instruct (Paper),N/A,N/A,N/A,N/A,N/A,N/A
Llama-3.1-8B-Instruct (Ours),+0.00,+1.00,+1.69,+0.01,+2.69,+2.69
Gemma-3-4B,-5.44,-0.37,-15.02,-1.93,-20.82,-22.76
Qwen3.5-9B-nothink,+1.45,-0.54,-1.64,-0.46,-0.87,-1.19


In [11]:
try:
    from IPython.display import Markdown
except ModuleNotFoundError:
    Markdown = None


def format_markdown_score(value, signed=False):
    if value is None:
        return "N/A"
    scaled = value * 100.0
    return f"{scaled:+.1f}" if signed else f"{scaled:.1f}"


def format_markdown_seconds(value, signed=False):
    if value is None:
        return "N/A"
    return f"{value:+.2f}" if signed else f"{value:.2f}"


def make_markdown_table(title, columns, rows, formatter):
    lines = [f"### {title}", ""]
    headers = [label for _, label in columns]
    aligns = [":---"] + ["---:"] * (len(columns) - 1)
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(aligns) + " |")
    for row in rows:
        cells = []
        for idx, (key, _) in enumerate(columns):
            value = row[key]
            if idx == 0:
                cells.append(str(value))
            else:
                cells.append(formatter(key, value))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


score_markdown_columns = [
    ("model", "Model"),
    ("before_qual", "Before QUAL"),
    ("before_leak", "Before LEAK"),
    ("after_qual", "After QUAL"),
    ("after_leak", "After LEAK"),
    ("delta_qual", "Delta QUAL"),
    ("delta_leak", "Delta LEAK"),
]

exclusion_markdown_columns = [
    ("model", "Model"),
    ("phase", "Stage"),
    ("qual_used", "QUAL Used Rows"),
    ("qual_excluded", "QUAL Negatives Excluded"),
    ("leak_used", "LEAK Used Rows"),
    ("leak_excluded", "LEAK Negatives Excluded"),
]

score_md = make_markdown_table(
    "QUAL / LEAK Summary (Markdown)",
    score_markdown_columns,
    score_rows,
    lambda key, value: format_markdown_score(value, signed=key.startswith("delta_")),
)

exclusion_md = make_markdown_table(
    "Negative-Score Filtering Summary (Markdown)",
    exclusion_markdown_columns,
    exclusion_rows,
    lambda key, value: str(value),
)

timing_before_md = make_markdown_table(
    "Timing Means By Model: Before (sec) (Markdown)",
    TIMING_COLUMNS,
    before_timing_rows,
    lambda key, value: format_markdown_seconds(value, signed=False),
)

timing_after_md = make_markdown_table(
    "Timing Means By Model: After (sec) (Markdown)",
    TIMING_COLUMNS,
    after_timing_rows,
    lambda key, value: format_markdown_seconds(value, signed=False),
)

timing_delta_md = make_markdown_table(
    "Timing Delta By Model: After - Before (sec) (Markdown)",
    TIMING_COLUMNS,
    delta_timing_rows,
    lambda key, value: format_markdown_seconds(value, signed=True),
)

markdown_bundle = "\n\n".join([
    score_md,
    exclusion_md,
    timing_before_md,
    timing_after_md,
    timing_delta_md,
])

if Markdown is not None:
    display(Markdown(markdown_bundle))
else:
    print(markdown_bundle)


### QUAL / LEAK Summary (Markdown)

| Model | Before QUAL | Before LEAK | After QUAL | After LEAK | Delta QUAL | Delta LEAK |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Llama-3.1-8B-Instruct (Paper) | 71.8 | 17.4 | 85.5 | 7.5 | +13.7 | -9.9 |
| Llama-3.1-8B-Instruct (Ours) | 76.7 | 29.3 | 91.1 | 13.4 | +14.4 | -15.9 |
| Gemma-3-4B | 70.6 | 62.3 | 67.2 | 20.5 | -3.4 | -41.8 |
| Qwen3.5-9B-nothink | 86.0 | 49.7 | 93.6 | 14.4 | +7.6 | -35.2 |

### Negative-Score Filtering Summary (Markdown)

| Model | Stage | QUAL Used Rows | QUAL Negatives Excluded | LEAK Used Rows | LEAK Negatives Excluded |
| :--- | ---: | ---: | ---: | ---: | ---: |
| Llama-3.1-8B-Instruct (Paper) | Before | N/A | N/A | N/A | N/A |
| Llama-3.1-8B-Instruct (Paper) | After | N/A | N/A | N/A | N/A |
| Llama-3.1-8B-Instruct (Ours) | Before | 236 | 0 | 236 | 0 |
| Llama-3.1-8B-Instruct (Ours) | After | 236 | 0 | 236 | 0 |
| Gemma-3-4B | Before | 235 | 0 | 235 | 0 |
| Gemma-3-4B | After | 235 | 0 | 235 | 0 |
| Qwen3.5-9B-nothink | Before | 236 | 1 | 236 | 1 |
| Qwen3.5-9B-nothink | After | 236 | 1 | 236 | 1 |

### Timing Means By Model: Before (sec) (Markdown)

| Model | Stage 1 | Stage 2a | Stage 2b | Judge | Pipeline | Total |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Llama-3.1-8B-Instruct (Paper) | N/A | N/A | N/A | N/A | N/A | N/A |
| Llama-3.1-8B-Instruct (Ours) | 3.12 | 6.90 | 6.60 | 11.54 | 16.62 | 28.16 |
| Gemma-3-4B | 7.91 | 7.49 | 18.78 | 11.79 | 34.18 | 45.96 |
| Qwen3.5-9B-nothink | 40.45 | 6.86 | 50.60 | 11.98 | 97.15 | 109.66 |

### Timing Means By Model: After (sec) (Markdown)

| Model | Stage 1 | Stage 2a | Stage 2b | Judge | Pipeline | Total |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Llama-3.1-8B-Instruct (Paper) | N/A | N/A | N/A | N/A | N/A | N/A |
| Llama-3.1-8B-Instruct (Ours) | 3.12 | 7.90 | 8.29 | 11.55 | 19.30 | 30.86 |
| Gemma-3-4B | 2.47 | 7.12 | 3.76 | 9.85 | 13.35 | 23.21 |
| Qwen3.5-9B-nothink | 41.89 | 6.32 | 48.96 | 11.52 | 96.28 | 108.47 |

### Timing Delta By Model: After - Before (sec) (Markdown)

| Model | Stage 1 | Stage 2a | Stage 2b | Judge | Pipeline | Total |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| Llama-3.1-8B-Instruct (Paper) | N/A | N/A | N/A | N/A | N/A | N/A |
| Llama-3.1-8B-Instruct (Ours) | +0.00 | +1.00 | +1.69 | +0.01 | +2.69 | +2.69 |
| Gemma-3-4B | -5.44 | -0.37 | -15.02 | -1.93 | -20.82 | -22.76 |
| Qwen3.5-9B-nothink | +1.45 | -0.54 | -1.64 | -0.46 | -0.87 | -1.19 |